# Hands-on exam 1 - part 1 (web scraping)
*University of Ljubljana, Faculty for computer and information science*

*Course: Introduction to data science*

This notebook should implement the web scraping of the [IMDB top 250 movies chart]( https://www.imdb.com/chart/top/).

Opening the link you should see the default chart view:

![IMDB default view](./imdb_top_charts.png)

**Switch to the detailed view by clicking the detailed view button as shown in the picture above. Your view should now have all relevant information you need for scraping:**

![IMDB default view](./imdb_top_charts_detailed.png)


The result is a JSON file containing an array of 250 entries, i.e.:

```json
[
    {
        "title": "The Shawshank Redemption",
        "year": 1994,
        "rating": 9.3,
        "directors": [
            "Frank Darabont"
        ],
        "movie_stars": [
            "Tim Robbins",
            "Morgan Freeman",
            "Bob Gunton"
        ]
    },
    {
        "title": "The Godfather",
        "year": 1972,
        "rating": 9.2,
        "directors": [
            "Francis Ford Coppola"
        ],
        "movie_stars": [
            "Marlon Brando",
            "Al Pacino",
            "James Caan"
        ]
    },
    ...
]
```

## Notes
- Do not worry about movie name translations.
- Notice that the ranking numbers are removed in the JSON file (i.e. `1. The Shawshank Redemption` is saved as `"title": "The Shawshank Redemption"`)
- "year" is an integer
- "rating" is a floating number
- each movie can have multiple directors and/or movie stars - save them as an array of strings.
- You do not have to close any popups (either about cookies or signing-in). The site is built in such a way that the content is loaded in the raw HTML.
- You do not have to scroll or paginate, all movies are loaded in the raw html.
- Do not worry of having exactly the same 250 entries as the provided failsafe JSON has - due to the nature of live chart ranking, these are subject to change.

In [1]:
# Import libraries and define global variables/parameters
URL = "https://www.imdb.com/chart/top/"
OUTPUT_JSON_FILE_NAME = "output.json"

Navigating to the site and switching to detailed view

Use the cell below to navigate to the url stored in the `URL` variable and switch to the detailed view by clicking the relevant button.

In [2]:
## Sample solution from me
from selenium import webdriver
from selenium.webdriver.common.by import By
from time import sleep
import json


driver = webdriver.Chrome()
driver.get(URL)
sleep(2)

# switch to detailed view which shows the movie stars too for each movie
driver.find_element(By.CSS_SELECTOR, "#list-view-option-detailed").click()
sleep(2)

Use your preferred method to produce a list of 250 HTML elements, each pertaining to 1 entry (movie) with all the data you need to scrape about it.

In [3]:
# at this point, the raw html has everything needed - so we could save it and use BS4 too.
movielist = driver.find_elements(By.CSS_SELECTOR, "li.ipc-metadata-list-summary-item")
assert len(movielist) == 250

Iterate over the list of HTML elements above, and scrape all needed data for each movie.

In [4]:
data = []
for movie in movielist:
    title = movie.find_element(By.CSS_SELECTOR, "h3").text
    # remove the leaderboard number from the title
    first_space_in_title = title.index(" ")
    title = title[first_space_in_title+1:]

    year = movie.find_elements(By.CSS_SELECTOR, "span.dli-title-metadata-item")[0].text
    rating = movie.find_element(By.CSS_SELECTOR, "span.ipc-rating-star--rating").text

    directors = movie.find_elements(By.CSS_SELECTOR, "a.dli-director-item")
    directors = [director.text for director in directors]
    movie_stars = movie.find_elements(By.CSS_SELECTOR, "a.dli-cast-item")
    movie_stars = [moive_star.text for moive_star in movie_stars]

    data.append({
        "title": title,
        "year": int(year),
        "rating": float(rating),
        "directors": directors,
        "movie_stars": movie_stars
    })
assert len(data) == 250
driver.quit()

Save the scraped data to the JSON file (name and path defined in the `OUTPUT_JSON_FILE_NAME` variable at the top of this notebook).

In [5]:
with open(OUTPUT_JSON_FILE_NAME, "w") as f:
    json.dump(data, f, indent=4)